# อธิบาย SCNN Pipeline — หา Lane Center

Notebook นี้อธิบาย pipeline ทีละขั้น:

```
Image  →  Preprocess (resize + normalize)  →  ResNet-18 + SpatialConv
                                                        ↓
                                            Softmax → Probability Map
                                                        ↓
                                          threshold (prob > 0.3) → pixels
                                                        ↓
                                              fit_lane() → polyfit x=f(y)
                                                        ↓
                                             lx, rx @ y_ref → center_norm
```

In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

PAD = '/home/peeradon/lka-carla-yolo/pytorch-auto-drive'
if PAD not in sys.path:
    sys.path.insert(0, PAD)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. โหลด SCNN model

In [ ]:
from utils.models.builder import MODELS

WEIGHTS = '/home/peeradon/lka-carla-yolo/pytorch-auto-drive/checkpoints/resnet18_scnn_lka/model.pt'
IMAGES  = {
    'clear': '/home/peeradon/lka-carla-yolo/Images/clear.png',
    'fog':   '/home/peeradon/lka-carla-yolo/Images/fog.png',
    'night': '/home/peeradon/lka-carla-yolo/Images/night.png',
    'rain':  '/home/peeradon/lka-carla-yolo/Images/rain.png',
}

# ── Parameters (same as scnn_node.py) ────────────────────────
IN_W        = 800
IN_H        = 288
PROB_THRESH = 0.3
Y_REF_RATIO = 0.85
NUM_CLASSES = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_cfg = dict(
    name='standard_segmentation_model',
    backbone_cfg=dict(
        name='predefined_resnet_backbone',
        backbone_name='resnet18',
        return_layer='layer4',
        pretrained=False,
        replace_stride_with_dilation=[False, True, True],
    ),
    reducer_cfg=dict(name='RESAReducer', in_channels=512, reduce=128),
    spatial_conv_cfg=dict(name='SpatialConv', num_channels=128),
    classifier_cfg=dict(name='DeepLabV1Head', in_channels=128,
                        num_classes=NUM_CLASSES, dilation=1),
    lane_classifier_cfg=dict(
        name='SimpleLaneExist',
        num_output=NUM_CLASSES - 1,
        flattened_size=(NUM_CLASSES * (IN_H // 16) * (IN_W // 16)),
    ),
)

net = MODELS.from_dict(model_cfg).to(device)
ckpt = torch.load(WEIGHTS, map_location=device)
net.load_state_dict(ckpt.get('model', ckpt))
net.eval()
print(f'SCNN loaded on {device}')
print(f'Backbone : ResNet-18')
print(f'Input    : {IN_W} x {IN_H} px')
print(f'Classes  : 0=background  1=left_marking  2=right_edge')

## 2. Preprocess — แปลงภาพก่อนส่งเข้าโมเดล

```
BGR image  →  RGB  →  resize 800×288  →  /255  →  normalize (ImageNet)  →  tensor
```

- mean = [0.485, 0.456, 0.406]  
- std  = [0.229, 0.224, 0.225]  (ค่ามาตรฐาน ImageNet)

In [ ]:
# ── ▼ Change this ▼ ───────────────────────────────────────────
WEATHER = 'night'   # 'clear' | 'fog' | 'night' | 'rain'
# ──────────────────────────────────────────────────────────────

img_bgr = cv2.imread(IMAGES[WEATHER])
H, W    = img_bgr.shape[:2]
print(f'Original image: {W}x{H} px')

mean = np.array([0.485, 0.456, 0.406], np.float32)
std  = np.array([0.229, 0.224, 0.225], np.float32)

rgb     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
resized = cv2.resize(rgb, (IN_W, IN_H)).astype(np.float32) / 255.0
normed  = (resized - mean) / std
x       = torch.from_numpy(normed.transpose(2, 0, 1)).unsqueeze(0).to(device)

print(f'Input tensor shape: {x.shape}  (batch=1, C=3, H={IN_H}, W={IN_W})')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(rgb)
axes[0].set_title(f'Original — {W}x{H} px [{WEATHER}]')
axes[0].axis('off')
axes[1].imshow(cv2.resize(rgb, (IN_W, IN_H)))
axes[1].set_title(f'Resized → {IN_W}x{IN_H} px (model input)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Inference — SCNN → Probability Map

โมเดลให้ output ดิบ (logits) ต่อ pixel → ผ่าน **Softmax** → ได้ probability 0.0–1.0 ต่อ class

```
logits (1, 3, H, W)  →  softmax(dim=1)  →  prob (3, H, W)
prob[0] = background
prob[1] = left_marking   ← สีเขียว
prob[2] = right_edge     ← สีน้ำเงิน
```

จากนั้น resize กลับเป็นขนาดภาพจริง (1600×900)

In [ ]:
with torch.no_grad():
    result  = net(x)
    logits  = result['out']                              # (1, 3, H_feat, W_feat)
    prob    = torch.softmax(logits, dim=1)[0].cpu().numpy()  # (3, H_feat, W_feat)

print(f'Feature map size (before resize): {prob.shape[1]}x{prob.shape[2]}')

left_prob  = cv2.resize(prob[1], (W, H))   # class 1 = left_marking
right_prob = cv2.resize(prob[2], (W, H))   # class 2 = right_edge

print(f'Probability map size (after resize): {W}x{H}')
print(f'left_prob  max={left_prob.max():.3f}  mean={left_prob.mean():.4f}')
print(f'right_prob max={right_prob.max():.3f}  mean={right_prob.mean():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Softmax Probability Map — raw output from SCNN', fontsize=13)

axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original image')
axes[0].axis('off')

im1 = axes[1].imshow(left_prob, cmap='Greens', vmin=0, vmax=1)
axes[1].set_title('left_prob (class 1 = left_marking)\nbright = high probability')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.03)

im2 = axes[2].imshow(right_prob, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title('right_prob (class 2 = right_edge)\nbright = high probability')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.03)

plt.tight_layout()
plt.show()

## 4. Threshold — กรองเอาเฉพาะ pixel ที่มั่นใจ

```
pixel ไหน prob > 0.3  →  นับเป็นเส้นนั้น
pixel ไหน prob ≤ 0.3  →  ทิ้ง (background)
```

In [ ]:
left_mask  = (left_prob  > PROB_THRESH).astype(np.uint8) * 255
right_mask = (right_prob > PROB_THRESH).astype(np.uint8) * 255

print(f'Threshold = {PROB_THRESH}')
print(f'left_mask  : {(left_mask  > 0).sum()} pixels passed')
print(f'right_mask : {(right_mask > 0).sum()} pixels passed')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(f'Threshold prob > {PROB_THRESH} → binary mask', fontsize=13)

axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original image')
axes[0].axis('off')

axes[1].imshow(left_mask, cmap='Greens')
axes[1].set_title(f'left_mask  (prob > {PROB_THRESH})\nwhite = left_marking pixel')
axes[1].axis('off')

axes[2].imshow(right_mask, cmap='Blues')
axes[2].set_title(f'right_mask (prob > {PROB_THRESH})\nwhite = right_edge pixel')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. `fit_lane` — mask pixels → polynomial line

เหมือน YOLO — เอา pixel สีขาวจาก mask ไป fit เส้นตรง **x = f(y)** ด้วย `np.polyfit`

```
np.where(prob > thresh)  →  (ys, xs)
np.polyfit(ys, xs, 1)    →  coefficients [m, b]
np.polyval(coeffs, y_ref) →  x ที่แถว y_ref
```

In [ ]:
def fit_lane(prob_map, x_min=0, x_max=None):
    """Fit line x=f(y) through pixels where prob > PROB_THRESH.
    Returns (coeffs, y_top, y_bot) or None."""
    if x_max is None:
        x_max = prob_map.shape[1]
    ys, xs = np.where(prob_map > PROB_THRESH)
    mask   = (xs >= x_min) & (xs < x_max)
    ys, xs = ys[mask], xs[mask]
    if len(xs) < 10:
        return None
    weights = prob_map[ys, xs]   # ใช้ prob เป็น weight → pixel มั่นใจมาก มีอิทธิพลมาก
    try:
        coeffs = np.polyfit(ys, xs, 1, w=weights)
    except np.linalg.LinAlgError:
        return None
    return coeffs, int(ys.min()), int(ys.max())


img_cx    = W // 2
y_ref     = int(H * Y_REF_RATIO)

left_fit  = fit_lane(left_prob,  x_min=0,      x_max=W * 3 // 4)
right_fit = fit_lane(right_prob, x_min=W // 4, x_max=W)

# Sanity check
if left_fit  is not None and np.polyval(left_fit[0],  y_ref) > img_cx: left_fit  = None
if right_fit is not None and np.polyval(right_fit[0], y_ref) < img_cx: right_fit = None

print(f'y_ref = {y_ref} px  (Y_REF_RATIO={Y_REF_RATIO})')
print(f'left_fit  : {"coeffs=" + str(np.round(left_fit[0], 4))  if left_fit  else "None"}')
print(f'right_fit : {"coeffs=" + str(np.round(right_fit[0], 4)) if right_fit else "None"}')

def draw_fit(img, fit, color):
    if fit is None:
        return
    coeffs, y_top, _ = fit
    ys = np.linspace(y_top, img.shape[0] - 1, 80).astype(int)
    xs = np.polyval(coeffs, ys).astype(int)
    for i in range(len(ys) - 1):
        if 0 <= xs[i] < img.shape[1] and 0 <= xs[i+1] < img.shape[1]:
            cv2.line(img, (xs[i], ys[i]), (xs[i+1], ys[i+1]), color, 4)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle('fit_lane — polyfit x=f(y) through prob > threshold pixels', fontsize=13)

for ax, (show_left, show_right, title) in zip(axes, [
    (True,  False, 'Left fit only'),
    (False, True,  'Right fit only'),
    (True,  True,  'Both + y_ref line'),
]):
    v = img_bgr.copy()

    if show_left and show_right:
        cv2.line(v, (0, y_ref), (W, y_ref), (0, 255, 255), 3)
        cv2.putText(v, f'y_ref={y_ref} (Y_REF_RATIO={Y_REF_RATIO})',
                    (10, y_ref - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

    if show_left:  draw_fit(v, left_fit,  (0, 255, 100))
    if show_right: draw_fit(v, right_fit, (0, 165, 255))

    if show_left and show_right and left_fit and right_fit:
        lx = int(np.polyval(left_fit[0],  y_ref))
        rx = int(np.polyval(right_fit[0], y_ref))
        cx = (lx + rx) // 2
        cv2.circle(v, (lx, y_ref), 10, (0, 255, 100),  -1)
        cv2.circle(v, (rx, y_ref), 10, (0, 165, 255),  -1)
        cv2.circle(v, (cx, y_ref), 14, (0, 0, 255),    -1)
        cv2.arrowedLine(v, (W // 2, y_ref), (cx, y_ref), (0, 255, 255), 2, tipLength=0.2)
        cv2.putText(v, f'lx={lx}', (lx+5, y_ref-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 100), 2)
        cv2.putText(v, f'rx={rx}', (rx+5, y_ref-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
        cv2.putText(v, f'cx={cx} → {cx/W:.3f}', (cx+5, y_ref+25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. คำนวณ center_norm — ผลลัพธ์สุดท้าย

```
lx          = polyval(left_coeffs,  y_ref)
rx          = polyval(right_coeffs, y_ref)
center_px   = (lx + rx) / 2
center_norm = center_px / W      →  publish /lka/lane_center  (0.0–1.0)
```

- center_norm = 0.5 → อยู่กลางภาพพอดี
- center_norm < 0.5 → เอียงซ้าย
- center_norm > 0.5 → เอียงขวา

In [ ]:
if left_fit is None or right_fit is None:
    print('NO DETECTION — detected = False → controller will brake')
else:
    lx          = float(np.polyval(left_fit[0],  y_ref))
    rx          = float(np.polyval(right_fit[0], y_ref))
    center_px   = (lx + rx) / 2.0
    center_norm = center_px / W
    lateral_err = center_px - (W / 2)

    print(f'[{WEATHER}]')
    print(f'  lx (left  @ y_ref) = {lx:.1f} px  →  {lx/W:.3f}')
    print(f'  rx (right @ y_ref) = {rx:.1f} px  →  {rx/W:.3f}')
    print(f'  center_px          = (lx + rx) / 2 = {center_px:.1f} px')
    print(f'  center_norm        = {center_px:.1f} / {W} = {center_norm:.3f}  → publish /lka/lane_center')
    print(f'  lateral_error      = center - image_center = {lateral_err:+.1f} px')
    print(f'  detected           = True')

    # visualize
    vis = img_bgr.copy()
    draw_fit(vis, left_fit,  (0, 255, 100))
    draw_fit(vis, right_fit, (0, 165, 255))
    lx_i = int(np.clip(lx, 0, W-1))
    rx_i = int(np.clip(rx, 0, W-1))
    cx_i = int(np.clip(center_px, 0, W-1))
    cv2.circle(vis, (lx_i, y_ref), 10, (0, 255, 100),  -1)
    cv2.circle(vis, (rx_i, y_ref), 10, (0, 165, 255),  -1)
    cv2.circle(vis, (cx_i, y_ref), 14, (0, 0, 255),    -1)
    cv2.line(vis, (W//2, y_ref-25), (W//2, y_ref+25), (255, 255, 0), 2)
    cv2.arrowedLine(vis, (W//2, y_ref), (cx_i, y_ref), (0, 255, 255), 2, tipLength=0.2)
    cv2.line(vis, (0, y_ref), (W, y_ref), (0, 255, 255), 1)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    for xv, col, ls, lbl in [
        (lx_i, 'tab:green',  '-',  f'lx = {lx:.0f} px  ({lx/W:.3f})'),
        (rx_i, 'tab:orange', '-',  f'rx = {rx:.0f} px  ({rx/W:.3f})'),
        (cx_i, 'tab:red',    '-',  f'center = (lx+rx)/2 = {center_px:.0f} px  →  {center_norm:.3f}'),
        (W//2, 'tab:cyan',   '--', f'image center = {W//2} px  (= 0.500)'),
    ]:
        ax.axvline(xv, color=col, lw=2, linestyle=ls, label=lbl)
    ax.set_title(f'SCNN center_norm = {center_norm:.3f}  [{WEATHER}]', fontsize=13)
    ax.legend(loc='upper left', fontsize=9, framealpha=0.85)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## 7. ทดสอบทุก 4 สภาพอากาศ

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.suptitle('SCNN — All 4 Weather Conditions', fontsize=14, fontweight='bold')

summary = []

for ax, weather in zip(axes.flat, ['clear', 'fog', 'night', 'rain']):
    img     = cv2.imread(IMAGES[weather])
    h, w    = img.shape[:2]
    yr      = int(h * Y_REF_RATIO)

    # preprocess
    rgb_    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res_    = cv2.resize(rgb_, (IN_W, IN_H)).astype(np.float32) / 255.0
    x_      = torch.from_numpy(((res_ - mean) / std).transpose(2, 0, 1)).unsqueeze(0).to(device)

    # inference
    with torch.no_grad():
        out_   = net(x_)['out']
        prob_  = torch.softmax(out_, dim=1)[0].cpu().numpy()
    lp = cv2.resize(prob_[1], (w, h))
    rp = cv2.resize(prob_[2], (w, h))

    # fit
    lf = fit_lane(lp, x_min=0,      x_max=w * 3 // 4)
    rf = fit_lane(rp, x_min=w // 4, x_max=w)
    if lf and np.polyval(lf[0], yr) > w // 2: lf = None
    if rf and np.polyval(rf[0], yr) < w // 2: rf = None

    vis = img.copy()
    cv2.line(vis, (0, yr), (w, yr), (0, 255, 255), 1)
    draw_fit(vis, lf, (0, 255, 100))
    draw_fit(vis, rf, (0, 165, 255))

    if lf and rf:
        lx_ = float(np.polyval(lf[0], yr))
        rx_ = float(np.polyval(rf[0], yr))
        cn  = (lx_ + rx_) / 2.0 / w
        err = (lx_ + rx_) / 2.0 - w / 2
        cx_ = int(np.clip((lx_ + rx_) / 2, 0, w-1))
        cv2.circle(vis, (cx_, yr), 14, (0, 0, 255), -1)
        cv2.arrowedLine(vis, (w//2, yr), (cx_, yr), (0, 255, 255), 2, tipLength=0.2)
        label = f'center={cn:.3f}  err={err:+.0f}px'
        summary.append((weather, cn, err, True))
    else:
        label = 'NO DETECTION'
        summary.append((weather, None, None, False))

    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{weather.upper()}  |  {label}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('\n-- Summary --------------------------------------------------')
print(f'{"Weather":8s}  {"center":>8s}  {"err(px)":>9s}  {"detected":>8s}')
for weather, cn, err, det in summary:
    cn_s  = f'{cn:.3f}'   if cn  is not None else 'MISS'
    err_s = f'{err:+.0f}' if err is not None else 'MISS'
    print(f'{weather:8s}  {cn_s:>8s}  {err_s:>9s}  {str(det):>8s}')